# Project 1: Student Performance Prediction System
**MCA III Semester – Lab Sheet-06: Mini Projects on Supervised Learning Applications**

**Problem statement:** Predict a student's final academic score (0–100) from attendance, internal marks, assignment score, daily study hours and previous semester percentage.

**Type of problem:** Supervised learning – **Regression**

**Workflow:** Load data → Preprocess → EDA → Train/Test split → Train & compare models → Evaluate → Visualize → Save model → Predict

In [ ]:
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 1. Load the Dataset

In [ ]:
student_df = pd.read_csv("student_performance_dataset.csv")
print("Shape:", student_df.shape)
student_df.head()

In [ ]:
student_df.info()

In [ ]:
student_df.describe().T

## 2. Data Preprocessing
### 2.1 Missing values and duplicates
Some records have missing values in `assignment_score`, `study_hours_per_day` and `prev_sem_percentage`. They will be filled with the **median** (robust to outliers).

In [ ]:
print("Missing values per column:")
print(student_df.isnull().sum())
print("\nDuplicate rows:", student_df.duplicated().sum())
student_df = student_df.drop_duplicates()

### 2.2 Outlier check
Box plots show whether any feature has extreme values.

In [ ]:
feature_columns = ["attendance_pct", "internal_marks", "assignment_score",
                   "study_hours_per_day", "prev_sem_percentage"]
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, column in zip(axes, feature_columns):
    sns.boxplot(y=student_df[column], ax=ax, color="skyblue")
    ax.set_title(column)
plt.tight_layout()
plt.show()

## 3. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(student_df["final_score"], bins=25, kde=True, color="teal")
plt.title("Distribution of Final Score")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(student_df.drop(columns="student_id").corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, column in zip(axes, ["internal_marks", "prev_sem_percentage", "study_hours_per_day"]):
    sns.scatterplot(data=student_df, x=column, y="final_score", alpha=0.5, ax=ax)
    ax.set_title(f"{column} vs final_score")
plt.tight_layout()
plt.show()

**Observation:** Internal marks and previous semester percentage show the strongest positive relationship with the final score; study hours and attendance also contribute.

## 4. Train / Test Split
80% of the data is used for training and 20% for testing. The imputer and scaler are placed **inside a pipeline** so they are fitted only on training data (prevents data leakage).

In [ ]:
X = student_df.drop(columns=["student_id", "final_score"])
y = student_df["final_score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print("Training samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])

## 5. Model Training and Comparison
Four regression algorithms are compared: **Linear Regression, Ridge Regression, Random Forest, Gradient Boosting**.

In [ ]:
def build_pipeline(model):
    """Median imputation -> standard scaling -> regression model."""
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", model),
    ])


candidate_models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}


def evaluate_model(name, pipeline):
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    mse = mean_squared_error(y_test, predictions)
    cv_r2 = cross_val_score(pipeline, X_train, y_train, cv=5, scoring="r2").mean()
    return {
        "Model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_test, predictions),
        "CV R2 (5-fold)": cv_r2,
    }


fitted_pipelines = {}
results = []
for model_name, model in candidate_models.items():
    pipeline = build_pipeline(model)
    results.append(evaluate_model(model_name, pipeline))
    fitted_pipelines[model_name] = pipeline

results_df = pd.DataFrame(results).sort_values("R2", ascending=False).reset_index(drop=True)
results_df.round(4)

### 5.1 Hyperparameter tuning (Random Forest)

In [ ]:
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 6, 10],
    "model__min_samples_leaf": [1, 3],
}
grid_search = GridSearchCV(build_pipeline(RandomForestRegressor(random_state=RANDOM_STATE)),
                           param_grid, cv=5, scoring="r2", n_jobs=-1)
grid_search.fit(X_train, y_train)
print("Best parameters:", grid_search.best_params_)

tuned_pipeline = grid_search.best_estimator_
fitted_pipelines["Random Forest (Tuned)"] = tuned_pipeline
tuned_predictions = tuned_pipeline.predict(X_test)
tuned_mse = mean_squared_error(y_test, tuned_predictions)
results_df.loc[len(results_df)] = {
    "Model": "Random Forest (Tuned)",
    "MAE": mean_absolute_error(y_test, tuned_predictions),
    "MSE": tuned_mse,
    "RMSE": np.sqrt(tuned_mse),
    "R2": r2_score(y_test, tuned_predictions),
    "CV R2 (5-fold)": grid_search.best_score_,
}
results_df = results_df.sort_values("R2", ascending=False).reset_index(drop=True)
results_df.round(4)

## 6. Visualization of Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=results_df, x="R2", y="Model", ax=axes[0], palette="viridis")
axes[0].set_title("R² Score (higher is better)")
sns.barplot(data=results_df, x="RMSE", y="Model", ax=axes[1], palette="magma")
axes[1].set_title("RMSE (lower is better)")
plt.tight_layout()
plt.show()

In [ ]:
best_model_name = results_df.loc[0, "Model"]
best_pipeline = fitted_pipelines[best_model_name]
best_predictions = best_pipeline.predict(X_test)
residuals = y_test - best_predictions
print("Best model:", best_model_name)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_test, best_predictions, alpha=0.6, color="royalblue")
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], "r--", label="Perfect prediction")
axes[0].set_xlabel("Actual Final Score")
axes[0].set_ylabel("Predicted Final Score")
axes[0].set_title("Actual vs Predicted")
axes[0].legend()

sns.histplot(residuals, bins=25, kde=True, ax=axes[1], color="darkorange")
axes[1].set_title("Residual Distribution")
axes[1].set_xlabel("Residual (Actual - Predicted)")
plt.tight_layout()
plt.show()

In [ ]:
final_estimator = best_pipeline.named_steps["model"]
if hasattr(final_estimator, "feature_importances_"):
    importance_values = final_estimator.feature_importances_
    importance_title = "Feature Importance"
else:
    importance_values = np.abs(final_estimator.coef_)
    importance_title = "Absolute Coefficient (scaled features)"

importance_df = pd.Series(importance_values, index=X.columns).sort_values()
importance_df.plot(kind="barh", figsize=(7, 4), color="seagreen")
plt.title(importance_title)
plt.show()

## 7. Interpretation of Evaluation Metrics
- **MAE** – average absolute difference between actual and predicted marks.
- **MSE / RMSE** – penalise large errors more strongly; RMSE is in the same unit as the score.
- **R² score** – proportion of variance in final score explained by the model (1.0 = perfect).
- **CV R²** – R² averaged across 5 folds of training data; close to the test R² means the model generalises well.

In [ ]:
best_row = results_df.iloc[0]
print(f"Best model : {best_row['Model']}")
print(f"R2 score   : {best_row['R2']:.3f} -> explains {best_row['R2'] * 100:.1f}% of the variation in final scores.")
print(f"MAE        : {best_row['MAE']:.2f} marks -> predictions are off by about {best_row['MAE']:.1f} marks on average.")
print(f"RMSE       : {best_row['RMSE']:.2f} marks")
gap = abs(best_row["R2"] - best_row["CV R2 (5-fold)"])
print(f"Test R2 vs CV R2 gap: {gap:.3f} -> {'good generalisation' if gap < 0.10 else 'possible over/under-fitting'}")

## 8. Save the Trained Model
The whole pipeline (imputer + scaler + model) is saved with Joblib, so new data can be fed in directly.

In [ ]:
MODEL_FILENAME = "student_performance_model.joblib"
joblib.dump(best_pipeline, MODEL_FILENAME)
print("Model saved as", MODEL_FILENAME)

## 9. Load the Model and Predict for a New Student

In [ ]:
loaded_model = joblib.load(MODEL_FILENAME)

new_student = pd.DataFrame([{
    "attendance_pct": 85,
    "internal_marks": 72,
    "assignment_score": 80,
    "study_hours_per_day": 4.0,
    "prev_sem_percentage": 74,
}])
predicted_score = loaded_model.predict(new_student)[0]
print(f"Predicted final score: {predicted_score:.1f} / 100")

## 10. Conclusion
- The complete workflow (preprocessing, training, evaluation, saving and prediction) was implemented using scikit-learn pipelines.
- Multiple algorithms were compared using MAE, MSE, RMSE and R²; the best performer was saved for deployment.
- Internal marks and previous semester percentage are the most influential predictors of final performance.